## Text-to-SQL Enterprise Agent (MVP)

This notebook builds a local SQLite database (`university_agent.db`) with a dummy University dataset and implements a safe Text-to-SQL pipeline.

**Architectural constraints covered:**
- Local SQLite only
- No RAG: static schema injection (DDL extraction)
- Strict separation: LLM generates SQL text only; Python executes
- Safety first: deterministic rule-based blocking of data-modifying SQL

**Before you run**
- Install dependencies: `pip install pandas google-generativeai`
- Set your API key (recommended): `export GEMINI_API_KEY="..."`


In [3]:
# Cell 1: Imports

from __future__ import annotations

import os
import re
import sqlite3
from dataclasses import dataclass
from typing import Any

import pandas as pd


In [17]:
import google.generativeai as genai
#need to update to google.genai -> resolve later

In [5]:
# Cell 2: Step 1 — Dummy dataset + SQLite database creation

def create_dummy_university_data(seed: int = 7) -> dict[str, pd.DataFrame]:
    """Create small, relational University tables.

    Trade-offs / report notes:
    - Synthetic data is ideal for demos and reproducibility.
    - The schema is intentionally small; static schema injection becomes costly
      as schemas grow (prompt length, latency, token cost).
    """
    rng = pd.Series(range(1, 21))  # 20 students

    students = pd.DataFrame(
        {
            "student_id": rng,
            "full_name": [f"Student {i:02d}" for i in rng],
            "email": [f"student{i:02d}@example.edu" for i in rng],
            "enrollment_year": [2022 + (i % 4) for i in rng],
            "major": [["CS", "DS", "IT", "Business", "Math"][i % 5] for i in rng],
        }
    )

    courses = pd.DataFrame(
        {
            "course_id": [101, 102, 103, 201, 202, 301],
            "course_code": ["CS101", "DS102", "IT103", "CS201", "BUS202", "MATH301"],
            "course_name": [
                "Intro to Programming",
                "Data Fundamentals",
                "Networks Basics",
                "Databases",
                "Business Analytics",
                "Linear Algebra",
            ],
            "credits": [6, 6, 6, 6, 6, 6],
        }
    )

    # Many-to-many between students and courses.
    # We keep it simple: each student takes 3 courses.
    enroll_rows: list[dict[str, Any]] = []
    grade_scale = ["HD", "D", "C", "P", "F"]

    for student_id in students["student_id"].tolist():
        course_choices = [
            courses.loc[(student_id + k) % len(courses), "course_id"]
            for k in (0, 2, 4)
        ]
        for course_id in course_choices:
            grade = grade_scale[(student_id + int(course_id)) % len(grade_scale)]
            score = int(55 + ((student_id * 7 + int(course_id)) % 46))  # 55..100
            enroll_rows.append(
                {
                    "student_id": int(student_id),
                    "course_id": int(course_id),
                    "semester": ["2024S1", "2024S2"][student_id % 2],
                    "grade": grade,
                    "score": score,
                }
            )

    grades = (
        pd.DataFrame(enroll_rows)
        .sort_values(["student_id", "course_id"])
        .reset_index(drop=True)
    )

    return {"students": students, "courses": courses, "grades": grades}


def write_university_db(db_path: str = "university_agent.db") -> str:
    """Create (or overwrite) a local SQLite DB and populate it.

    Trade-offs / report notes:
    - For MVP speed, we use pandas `to_sql`.
    - We add PK/FK constraints after writing using a rename/recreate pattern
      (SQLite has limited ALTER TABLE support).
    """
    data = create_dummy_university_data()

    if os.path.exists(db_path):
        os.remove(db_path)

    with sqlite3.connect(db_path) as conn:
        conn.execute("PRAGMA foreign_keys = ON;")

        data["students"].to_sql("students", conn, index=False)
        data["courses"].to_sql("courses", conn, index=False)
        data["grades"].to_sql("grades", conn, index=False)

        conn.executescript(
            """
            PRAGMA foreign_keys = OFF;

            ALTER TABLE students RENAME TO students_old;
            CREATE TABLE students (
                student_id INTEGER PRIMARY KEY,
                full_name TEXT NOT NULL,
                email TEXT NOT NULL UNIQUE,
                enrollment_year INTEGER NOT NULL,
                major TEXT NOT NULL
            );
            INSERT INTO students SELECT * FROM students_old;
            DROP TABLE students_old;

            ALTER TABLE courses RENAME TO courses_old;
            CREATE TABLE courses (
                course_id INTEGER PRIMARY KEY,
                course_code TEXT NOT NULL UNIQUE,
                course_name TEXT NOT NULL,
                credits INTEGER NOT NULL
            );
            INSERT INTO courses SELECT * FROM courses_old;
            DROP TABLE courses_old;

            ALTER TABLE grades RENAME TO grades_old;
            CREATE TABLE grades (
                student_id INTEGER NOT NULL,
                course_id INTEGER NOT NULL,
                semester TEXT NOT NULL,
                grade TEXT NOT NULL,
                score INTEGER NOT NULL,
                PRIMARY KEY (student_id, course_id, semester),
                FOREIGN KEY (student_id) REFERENCES students(student_id),
                FOREIGN KEY (course_id) REFERENCES courses(course_id)
            );
            INSERT INTO grades SELECT * FROM grades_old;
            DROP TABLE grades_old;

            PRAGMA foreign_keys = ON;
            """
        )

    return db_path


# Create the DB now (idempotent-ish for notebooks)
db_path = write_university_db("university_agent.db")
print("DB created at:", db_path)


DB created at: university_agent.db


In [6]:
# Cell 3: Step 2 — Schema extraction (Static Schema Injection)

def get_schema(db_path: str) -> str:
    """Extract CREATE TABLE statements for all user tables in SQLite.

    Trade-offs / report notes:
    - Static injection is simple and deterministic (no retrieval system).
    - It scales poorly: large schemas increase prompt length, latency, and cost.
    - For a 1-week MVP and small schemas, it is an excellent baseline.
    """
    with sqlite3.connect(db_path) as conn:
        rows = conn.execute(
            """
            SELECT sql
            FROM sqlite_master
            WHERE type='table'
              AND name NOT LIKE 'sqlite_%'
              AND sql IS NOT NULL
            ORDER BY name;
            """
        ).fetchall()

    ddl_statements = [r[0].strip().rstrip(";") + ";" for r in rows]
    return "\n\n".join(ddl_statements)


schema_text = get_schema(db_path)
print(schema_text)


CREATE TABLE courses (
                course_id INTEGER PRIMARY KEY,
                course_code TEXT NOT NULL UNIQUE,
                course_name TEXT NOT NULL,
                credits INTEGER NOT NULL
            );

CREATE TABLE grades (
                student_id INTEGER NOT NULL,
                course_id INTEGER NOT NULL,
                semester TEXT NOT NULL,
                grade TEXT NOT NULL,
                score INTEGER NOT NULL,
                PRIMARY KEY (student_id, course_id, semester),
                FOREIGN KEY (student_id) REFERENCES students(student_id),
                FOREIGN KEY (course_id) REFERENCES courses(course_id)
            );

CREATE TABLE students (
                student_id INTEGER PRIMARY KEY,
                full_name TEXT NOT NULL,
                email TEXT NOT NULL UNIQUE,
                enrollment_year INTEGER NOT NULL,
                major TEXT NOT NULL
            );


In [ ]:
# Cell 4: Step 2 — LLM SQL generation (Gemini)

SQL_TRANSLATION_SYSTEM_PROMPT = """\
You are an expert data analyst and SQL translator.
Your ONLY job is to translate the user's question into a SINGLE SQLite SELECT query.

Rules (must follow):
- Output ONLY the SQL query text. No markdown fences, no explanations.
- Use ONLY the tables and columns that exist in the provided schema.
- Generate READ-ONLY SQL: SELECT queries only.
- Do NOT use any data-modifying statements: INSERT, UPDATE, DELETE, DROP, ALTER, CREATE, REPLACE, TRUNCATE, VACUUM, PRAGMA, ATTACH, DETACH.
- Do NOT reference sqlite_master or any internal SQLite tables.
- Prefer simple SQL compatible with SQLite.

If the question cannot be answered using the schema, output exactly:
SELECT 'UNANSWERABLE_WITH_GIVEN_SCHEMA' AS error;
"""


def generate_sql(user_question: str, schema_text: str, *, model_name: str = "gemini-1.5-flash") -> str:
    """Call Gemini to generate SQL from a question + schema.

    Strict separation guarantee:
    - The LLM sees ONLY schema DDL (no row data).
    - The LLM returns ONLY SQL text; Python decides if/when to execute.

    Trade-offs / report notes:
    - LLM output can be invalid SQL or reference non-existent columns.
      We handle this with execution-time error handling and safety checks.
    - Prompt-injection risk exists (e.g., user asks to DROP TABLE). We still
      deterministically block dangerous tokens before execution.
    """
    # Lazy import so the rest of the notebook can run without the dependency installed.
    # import google.generativeai as genai  # type: ignore

    # Use an env var in real use. Never hardcode real secrets into Git.
    genai.configure(api_key=os.environ.get("GEMINI_API_KEY", "YOUR_API_KEY_HERE"))

    model = genai.GenerativeModel(
        model_name, 
        system_instruction=SQL_TRANSLATION_SYSTEM_PROMPT)

    prompt = f"""\
### SQLite schema (DDL)
{schema_text}

### User question
{user_question}
"""

    try:
      resp = model.generate_content(
          prompt,
          generation_config={
              "temperature": 0.0,  # reproducibility / less randomness
              "max_output_tokens": 512,
          },
          # system_instruction=SQL_TRANSLATION_SYSTEM_PROMPT,
      )

      sql = (resp.text or "").strip()

      # Defensive cleanup: strip code fences if the model returns them.
      sql = re.sub(r"^```(?:sql)?\s*", "", sql, flags=re.IGNORECASE).strip()
      sql = re.sub(r"\s*```$", "", sql).strip()

      return sql
    except Exception as e:
      return f"Error connecting to Gemini API: {e}"


In [20]:
# Cell 5: Step 2 — Deterministic safety gate (block dangerous SQL)

_DANGEROUS_SQL_PATTERN = re.compile(
    r"""
    (?ix)                       # i=case-insensitive, x=verbose
    \b(
        insert|update|delete|drop|alter|create|replace|truncate|
        vacuum|pragma|attach|detach|reindex|analyze|
        begin|commit|rollback|savepoint|release
    )\b
    """.strip()
)


def is_safe_query(sql_string: str) -> bool:
    """Deterministic rule-based safety filter.

    Security stance:
    - Fail closed. Allow only read-only queries.
    - Block anything that looks like write/DDL/transaction SQL.

    Trade-offs / report notes:
    - This conservative keyword policy may create false positives.
      For an MVP, that's acceptable because safety > completeness.
    - A SQL parser can help, but still requires careful policy design.
    """
    if not sql_string or not sql_string.strip():
        return False

    s = sql_string.strip().rstrip(";").strip()

    # Must start with SELECT or WITH (CTE) for a read-only query.
    if not re.match(r"(?is)^(select|with)\b", s):
        return False

    # Block dangerous tokens anywhere.
    if _DANGEROUS_SQL_PATTERN.search(s) is not None:
        return False

    # Block internal SQLite metadata tables.
    if re.search(r"(?is)\bsqlite_master\b|\bsqlite_schema\b", s):
        return False

    return True


In [25]:
# Cell 6: Step 2 — SQLite execution (Python runs SQL, not the LLM)

@dataclass(frozen=True)
class QueryResult:
    columns: list[str]
    rows: list[tuple[Any, ...]]


def execute_query(db_path: str, sql_string: str) -> QueryResult:
    """Execute a SQL query and fetch all rows.

    Trade-offs / report notes:
    - `fetchall()` is fine for MVP and small datasets.
    - For large results, you would paginate or stream.
    """
    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        cur = conn.execute(sql_string)
        rows = cur.fetchall()
        columns = [d[0] for d in cur.description] if cur.description else []
        return QueryResult(columns=columns, rows=[tuple(r) for r in rows])


In [34]:
# Cell 7: Step 3 — Master wrapper function

def ask_database(
    question: str,
    *,
    db_path: str = "university_agent.db",
    model_name: str = "gemini-2.5-flash",
) -> QueryResult:
    """End-to-end agent wrapper:
    schema extraction -> LLM SQL generation -> safety validation -> execution.

    Error handling principles:
    - Fail closed: unsafe or invalid SQL is not executed.
    - Notebook-friendly errors: return a 1-row result with an error message.
      (Alternative design: raise exceptions and let a UI handle them.)
    """
    if not os.path.exists(db_path):
        # Notebook convenience: auto-create if missing.
        write_university_db(db_path)

    try:
        schema_text = get_schema(db_path)
        sql = generate_sql(question, schema_text, model_name=model_name)

        if "UNANSWERABLE_WITH_GIVEN_SCHEMA" in sql:
            return QueryResult(columns=["error"], rows=[("UNANSWERABLE_WITH_GIVEN_SCHEMA",)])

        if not is_safe_query(sql):
            return QueryResult(columns=["error", "sql"], rows=[("BLOCKED_UNSAFE_SQL", sql)])

        return execute_query(db_path, sql)

    except Exception as e:
        # In production you'd log stack traces, add retries/timeouts, etc.
        return QueryResult(columns=["error"], rows=[(f"{type(e).__name__}: {e}",)])


In [35]:
# Cell 8: Quick tests
# Note: These tests require a working Gemini key if you want the LLM-generated SQL.
# If GEMINI_API_KEY is not set, you'll see a ModuleNotFoundError (dependency missing)
# or auth-related error (key missing/invalid).

questions = [
    "How many students are enrolled in each major?",
    "List the top 5 students by average score.",
    "Show each course and the number of distinct students who took it.",
    "DROP TABLE students;",  # should be blocked by safety gate
]

for q in questions:
    print("\nQ:", q)
    result = ask_database(q, db_path=db_path)
    print("Columns:", result.columns)
    print("Rows (first 10):", result.rows[:10])



Q: How many students are enrolled in each major?
Columns: ['major', 'COUNT(student_id)']
Rows (first 10): [('Business', 4), ('CS', 4), ('DS', 4), ('IT', 4), ('Math', 4)]

Q: List the top 5 students by average score.
Columns: ['full_name', 'average_score']
Rows (first 10): [('Student 10', 91.66666666666667), ('Student 09', 89.33333333333333), ('Student 16', 87.66666666666667), ('Student 18', 86.33333333333333), ('Student 15', 85.33333333333333)]

Q: Show each course and the number of distinct students who took it.
Columns: ['course_name', 'COUNT(DISTINCT T2.student_id)']
Rows (first 10): [('Intro to Programming', 10), ('Data Fundamentals', 10), ('Networks Basics', 10), ('Databases', 10), ('Business Analytics', 10), ('Linear Algebra', 10)]

Q: DROP TABLE students;
Columns: ['error', 'sql']
Rows (first 10): [('BLOCKED_UNSAFE_SQL', 'Error connecting to Gemini API: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: 